# Module 7.5 — Interfacing with C

### Python Mastery · Track 7: Performance, Internals & C Extensions

When Python is too slow for a hot loop, or when you need to use an existing C library, you can drop down to C. This module covers four approaches, from easiest to most involved: calling shared libraries with `ctypes`, the friendlier CFFI, accelerating Python with Cython, and writing a minimal CPython C extension by hand.

**How to use this notebook**

- Read each explanation, then run the code cell beneath it.
- This environment has a C compiler, Cython, and CFFI, so the examples genuinely compile and run real native code. Compilation steps write source files and invoke the compiler with `!`. If you run this elsewhere without a compiler, those steps will not build, but the explanations still apply.
- Attempt the exercises before consulting the solutions.

### Learning objectives

After completing this module you will be able to:

1. Call functions in a shared C library with `ctypes`.
2. Use CFFI to bind C code more conveniently.
3. Accelerate a Python function with Cython and measure the gain.
4. Describe the structure of a CPython C extension.
5. Choose an appropriate approach for a given need.

**Prerequisites:** Tracks 1 to 6, and Modules 7.1 to 7.4.

---

## Part 1 · `ctypes`: Calling Shared Libraries

`ctypes` is in the standard library and needs no compilation step of its own: it loads an existing shared library and calls its functions directly from Python. It is the quickest way to use a C library you already have. You specify each function's argument and return types so Python can convert values correctly.

To demonstrate against real compiled code, we first write and compile a tiny C library, then call it.

In [ ]:
%%writefile mathlib.c
// A tiny C library with two functions.
long add(long a, long b) {
    return a + b;
}

double circle_area(double radius) {
    return 3.14159265358979 * radius * radius;
}

In [ ]:
# Compile the C source into a shared library (.so on Linux).
!gcc -shared -fPIC -o libmath.so mathlib.c && echo "compiled libmath.so"

In [ ]:
import ctypes

# Load the shared library we just built.
lib = ctypes.CDLL("./libmath.so")

# Declare the signatures so ctypes converts arguments and results correctly.
lib.add.argtypes = [ctypes.c_long, ctypes.c_long]
lib.add.restype = ctypes.c_long

lib.circle_area.argtypes = [ctypes.c_double]
lib.circle_area.restype = ctypes.c_double

# Call the C functions as if they were Python functions.
print("add(20, 22) =", lib.add(20, 22))
print("circle_area(2.0) =", round(lib.circle_area(2.0), 4))

## Part 2 · CFFI: A Friendlier Binding

`ctypes` works but is verbose, and declaring signatures by hand is error-prone. **CFFI** (C Foreign Function Interface) lets you paste C declarations directly and call the functions more naturally. In its simple "ABI" mode it loads a library and reads ordinary C declarations, which is easier to get right than manual `ctypes` annotations.

In [ ]:
from cffi import FFI

ffi = FFI()
# Paste the C declarations exactly as they appear in a header.
ffi.cdef("""
    long add(long a, long b);
    double circle_area(double radius);
""")

# Open the same shared library built in Part 1.
lib = ffi.dlopen("./libmath.so")

print("add(100, 23) =", lib.add(100, 23))
print("circle_area(3.0) =", round(lib.circle_area(3.0), 4))
print("CFFI read the C declarations directly, with no manual type annotations")

## Part 3 · Cython: Python That Compiles to C

**Cython** is a superset of Python that compiles to C. You can take ordinary Python and add static type declarations (`cdef int`, and so on); Cython then generates and compiles C code that can run far faster for numeric, loop-heavy work. It is often the best balance of effort and speed-up, since you write almost-Python rather than raw C.

The cell below writes a Cython module with a typed function and a pure-Python equivalent for comparison, then builds it.

In [ ]:
%%writefile fastmath.pyx
# A Cython module (.pyx). The cdef type declarations let Cython generate tight C.
# Note: i is declared 'long', not 'int', so that i * i is computed in 64-bit
# arithmetic. With a 32-bit int, i * i would overflow for large i and give a
# wrong result; matching the C types to the value range is part of using Cython.
def sum_squares_cython(long n):
    cdef long total = 0
    cdef long i
    for i in range(n):
        total += i * i
    return total

In [ ]:
%%writefile setup_cython.py
from setuptools import setup
from Cython.Build import cythonize

setup(
    name="fastmath",
    ext_modules=cythonize("fastmath.pyx", language_level=3),
)

In [ ]:
# Build the Cython module in place. This compiles the generated C to a .so.
!python setup_cython.py build_ext --inplace 2>/dev/null | tail -2
import glob
print("built module:", glob.glob("fastmath*.so"))

In [ ]:
import timeit
import fastmath          # the compiled Cython module

# A pure-Python equivalent for a fair comparison.
def sum_squares_python(n):
    total = 0
    for i in range(n):
        total += i * i
    return total

n = 1_000_000
# Confirm identical results before comparing speed.
assert sum_squares_python(n) == fastmath.sum_squares_cython(n)

py_time = timeit.timeit(lambda: sum_squares_python(n), number=5)
cy_time = timeit.timeit(lambda: fastmath.sum_squares_cython(n), number=5)
print(f"pure Python: {py_time:.4f}s")
print(f"Cython:      {cy_time:.4f}s")
print(f"speed-up: about {py_time/cy_time:.0f}x for the same result")

The typed Cython version runs substantially faster than pure Python on this loop, because the arithmetic happens in compiled C with machine integers rather than Python objects. Cython shines exactly here: tight numeric loops that are slow in pure Python.

One caution comes with that speed: C integer types have fixed ranges, unlike Python's arbitrary-precision integers. In the loop above, the loop variable is declared `long` (not `int`) precisely so that `i * i` is computed in 64-bit arithmetic; a 32-bit `int` would overflow for large `i` and silently produce a wrong answer. When you add C types to Cython code, you take on responsibility for choosing types wide enough for the values involved.

## Part 4 · A Minimal CPython C Extension

At the lowest level you can write a C extension directly against the **CPython C-API**. This is the most work but gives full control and is how the interpreter's own modules and libraries like NumPy are built. An extension defines C functions that accept and return `PyObject*` values, a method table, and a module definition, then exposes an initialisation function. We compile and import a real one.

In [ ]:
%%writefile greetmodule.c
#define PY_SSIZE_T_CLEAN
#include <Python.h>

// A C function exposed to Python. It parses one string argument and returns a string.
static PyObject* greet_hello(PyObject* self, PyObject* args) {
    const char* name;
    if (!PyArg_ParseTuple(args, "s", &name)) {   // "s" = one C string
        return NULL;                              // parsing failed; exception is set
    }
    return PyUnicode_FromFormat("Hello, %s, from C!", name);
}

// A second function that adds two integers, releasing nothing fancy.
static PyObject* greet_add(PyObject* self, PyObject* args) {
    long a, b;
    if (!PyArg_ParseTuple(args, "ll", &a, &b)) {  // "ll" = two C longs
        return NULL;
    }
    return PyLong_FromLong(a + b);
}

// The method table: maps Python names to C functions.
static PyMethodDef GreetMethods[] = {
    {"hello", greet_hello, METH_VARARGS, "Greet someone from C."},
    {"add",   greet_add,   METH_VARARGS, "Add two integers in C."},
    {NULL, NULL, 0, NULL}                          // sentinel terminator
};

// The module definition.
static struct PyModuleDef greetmodule = {
    PyModuleDef_HEAD_INIT, "greet", "A tiny C extension.", -1, GreetMethods
};

// The initialisation function: named PyInit_<modulename>.
PyMODINIT_FUNC PyInit_greet(void) {
    return PyModule_Create(&greetmodule);
}

In [ ]:
%%writefile setup_ext.py
from setuptools import setup, Extension

setup(
    name="greet",
    ext_modules=[Extension("greet", ["greetmodule.c"])],
)

In [ ]:
# Compile the C extension in place.
!python setup_ext.py build_ext --inplace 2>/dev/null | tail -2
import glob
print("built extension:", glob.glob("greet*.so"))

In [ ]:
import greet          # our compiled C extension

print(greet.hello("Asha"))
print("greet.add(40, 2) =", greet.add(40, 2))
print("docstring from C:", greet.hello.__doc__)

Every line above ran genuine compiled C. The structure, C functions taking `PyObject*`, a method table, a module definition, and a `PyInit_` function, is the same pattern used throughout CPython itself. Recall from Module 7.3 that such a C function can release the GIL (`Py_BEGIN_ALLOW_THREADS`) around heavy work, enabling real parallelism.

## Part 5 · Choosing an Approach

Each tool fits a different need:

- **`ctypes`** when you have an existing shared library and want to call it with no build step. Quick, standard-library, but verbose and manual.
- **CFFI** when binding C and you want cleaner declarations or to build a faster compiled binding. Friendlier than `ctypes`.
- **Cython** when you want to speed up your own Python loops with minimal rewriting. Often the best effort-to-speed ratio.
- **A C extension** when you need full control or are wrapping a substantial C codebase. The most work, the most power.

In practice, reach for the simplest tool that solves the problem, and only drop lower when you must. And remember the prior advice: profile first (Module 7.1) and choose a better algorithm before reaching for C at all.

In [ ]:
options = [
    ("ctypes",      "call an existing .so/.dll", "stdlib, no build", "verbose, manual types"),
    ("CFFI",        "bind C cleanly",            "reads C declarations", "extra dependency"),
    ("Cython",      "speed up your own loops",   "near-Python, big gains", "needs a build step"),
    ("C extension", "full control / wrap C",     "maximum power",        "most effort, raw C"),
]
print(f"{'tool':12} {'use when':28} {'strength':24} {'cost'}")
for tool, use, strength, cost in options:
    print(f"{tool:12} {use:28} {strength:24} {cost}")

---

## Worked Examples

| Examples | Level |
|---|---|
| 1 and 2 | Easy |
| 3 and 4 | Medium |
| 5 and 6 | Difficult |

### Example 1 — Calling a C function with ctypes (Easy)

In [ ]:
import ctypes
# Reuse the library compiled in Part 1.
lib = ctypes.CDLL("./libmath.so")
lib.add.argtypes = [ctypes.c_long, ctypes.c_long]
lib.add.restype = ctypes.c_long
print("add(7, 8) =", lib.add(7, 8))

### Example 2 — Using the standard C math library via ctypes (Easy)

In [ ]:
import ctypes, ctypes.util

# Load the system C math library and call sqrt directly.
libm_name = ctypes.util.find_library("m")
libm = ctypes.CDLL(libm_name)
libm.sqrt.argtypes = [ctypes.c_double]
libm.sqrt.restype = ctypes.c_double
print("C sqrt(144) =", libm.sqrt(144.0))

### Example 3 — A CFFI binding (Medium)

In [ ]:
from cffi import FFI
ffi = FFI()
ffi.cdef("long add(long a, long b);")
lib = ffi.dlopen("./libmath.so")
print("via CFFI, add(15, 27) =", lib.add(15, 27))

### Example 4 — A Cython speed comparison (Medium)

In [ ]:
import timeit, fastmath

def py_version(n):
    total = 0
    for i in range(n):
        total += i * i
    return total

n = 500_000
assert py_version(n) == fastmath.sum_squares_cython(n)
print(f"python: {timeit.timeit(lambda: py_version(n), number=5):.4f}s")
print(f"cython: {timeit.timeit(lambda: fastmath.sum_squares_cython(n), number=5):.4f}s")

### Example 5 — Calling our hand-written C extension (Difficult)

In [ ]:
import greet

# Both functions come from compiled C code.
names = ["Asha", "Ben", "Chen"]
for name in names:
    print(greet.hello(name))
print("sum via C:", greet.add(1000, 234))

### Example 6 — Passing arrays to C with ctypes (Difficult)

In [ ]:
%%writefile arraylib.c
// Sum an array of doubles passed from Python.
double sum_array(double* values, int count) {
    double total = 0.0;
    for (int i = 0; i < count; i++) {
        total += values[i];
    }
    return total;
}

In [ ]:
!gcc -shared -fPIC -o libarray.so arraylib.c && echo "compiled libarray.so"

In [ ]:
import ctypes

lib = ctypes.CDLL("./libarray.so")
lib.sum_array.argtypes = [ctypes.POINTER(ctypes.c_double), ctypes.c_int]
lib.sum_array.restype = ctypes.c_double

# Build a C array from Python data and pass it with its length.
values = [1.5, 2.5, 3.0, 4.0]
c_array = (ctypes.c_double * len(values))(*values)
result = lib.sum_array(c_array, len(values))
print("sum computed in C:", result)

---

## Exercises

The shared library `libmath.so` and the modules `fastmath` and `greet` were built earlier in this notebook; the exercises reuse them.

**Exercise 1 (Easy).** Using `ctypes`, load `./libmath.so`, declare `circle_area` (one double in, one double out), and print the area for radius 5.

In [ ]:
# Your solution here


**Exercise 2 (Easy).** Using `ctypes.util.find_library`, load the system math library and call `pow(2, 10)` through it (declare argtypes as two doubles, restype double).

In [ ]:
# Your solution here


**Exercise 3 (Medium).** Using CFFI, declare and call `circle_area` from `./libmath.so` for radius 1.0, and print the result.

In [ ]:
# Your solution here


**Exercise 4 (Medium).** Compare the Cython `fastmath.sum_squares_cython` against a pure-Python loop for `n = 300_000`, asserting equal results and printing both timings.

In [ ]:
# Your solution here


**Exercise 5 (Difficult).** Call the hand-written `greet` extension: greet three different names and compute `greet.add` of two numbers, printing each result.

In [ ]:
# Your solution here


---

## Solutions

**Solution 1**

In [ ]:
import ctypes
lib = ctypes.CDLL("./libmath.so")
lib.circle_area.argtypes = [ctypes.c_double]
lib.circle_area.restype = ctypes.c_double
print("area at r=5:", round(lib.circle_area(5.0), 4))

**Solution 2**

In [ ]:
import ctypes, ctypes.util
libm = ctypes.CDLL(ctypes.util.find_library("m"))
libm.pow.argtypes = [ctypes.c_double, ctypes.c_double]
libm.pow.restype = ctypes.c_double
print("pow(2, 10) =", libm.pow(2.0, 10.0))

**Solution 3**

In [ ]:
from cffi import FFI
ffi = FFI()
ffi.cdef("double circle_area(double radius);")
lib = ffi.dlopen("./libmath.so")
print("area at r=1:", round(lib.circle_area(1.0), 5))

**Solution 4**

In [ ]:
import timeit, fastmath
def py(n):
    t = 0
    for i in range(n):
        t += i * i
    return t
n = 300_000
assert py(n) == fastmath.sum_squares_cython(n)
print(f"python: {timeit.timeit(lambda: py(n), number=5):.4f}s")
print(f"cython: {timeit.timeit(lambda: fastmath.sum_squares_cython(n), number=5):.4f}s")

**Solution 5**

In [ ]:
import greet
for name in ["Dev", "Ela", "Fay"]:
    print(greet.hello(name))
print("add:", greet.add(11, 31))

---

## Summary and Key Points

- `ctypes` (standard library) loads an existing shared library and calls its functions after you declare their argument and return types; quickest for using a C library you already have, but verbose.
- CFFI binds C more conveniently by reading ordinary C declarations directly, avoiding manual type annotations.
- Cython compiles a Python superset to C; adding static type declarations to hot numeric loops yields large speed-ups for near-Python effort, at the cost of a build step.
- A CPython C extension is written against the C-API with `PyObject*` functions, a method table, a module definition, and a `PyInit_` function; it is the most work and the most powerful, and is how CPython modules and libraries like NumPy are built (and where the GIL can be released around heavy work).
- Choose the simplest tool that fits, and always profile and improve the algorithm first (Module 7.1) before reaching for C.

### Next module: 7.6 — Alternative Runtimes & JIT

The next module surveys other ways to run Python faster: the PyPy just-in-time compiler, and the in-interpreter JIT and specialisation work in modern CPython.